# Day 17 — GroupBy & Aggregation
### Python for Data Science · Module 1 · Topic 1.16

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

**Session length:** 2 hours
**Format:** 90 min concepts + live coding · 30 min practice

| # | What we cover | Time |
|---|---|---|
| 1 | Split, apply, combine — the mental model | 20 min |
| 2 | Aggregating — one function, or many | 25 min |
| 3 | Grouping by two columns | 20 min |
| 4 | **`transform`, and the missing-key pitfall** | 20 min |
| 5 | Mini build: a department report | 5 min |
| 6 | **Practice notebook (separate file)** | 30 min |

> **The heuristic worth learning today.** Almost every real question about data is a grouped
> question — sales *per* region, average delay *per* airline, churn rate *per* plan. **Once
> you hear the word "per", you are looking at a groupby**, and the answer is usually one line.

---
## 0. The loop we promised to replace

This is what you wrote in Day 15's practice:

```python
for city in df["city"].unique():
    mask = df["city"] == city
    print(city, df.loc[mask, "python"].mean())
```

Four lines, a loop, and manual printing. Here is today's version.

In [6]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    "name":   ["Ravi", "Sara", "Amit", "Neha", "Kiran", "Priya"],
    "city":   ["Pune", "Mumbai", "Delhi", "Pune", "Delhi", "Mumbai"],
    "grade":  ["A", "A", "C", "B", "B", "B"],
    "python": [88, 91, 45, 67, 72, 58],
    "stats":  [71, 84, 38, 73, 69, 62],
})

df.groupby("grade")["python"].std()

grade
A    2.121320
B    7.094599
C         NaN
Name: python, dtype: float64

In [4]:
df

,name,city,grade,python,stats
0,Ravi,Pune,A,88,71
1,Sara,Mumbai,A,91,84
2,Amit,Delhi,C,45,38
3,Neha,Pune,B,67,73
4,Kiran,Delhi,B,72,69
5,Priya,Mumbai,B,58,62


One line — and the result is a Series you can sort, filter or plot. It extends to two
subjects, three statistics, or two grouping columns without getting any longer.

---
# 1. Split → apply → combine

```
df.groupby("city")["python"].mean()

  1. SPLIT                2. APPLY           3. COMBINE
  Pune:   88, 67   ->     mean = 77.5   ->   city
  Mumbai: 91, 58   ->     mean = 74.5        Delhi     58.5
  Delhi:  45, 72   ->     mean = 58.5        Mumbai    74.5
                                             Pune      77.5
```

**You never see step 1.** pandas splits, applies and recombines in one operation.

Two things to notice in the result: it is sorted by **group key** (not by value), and the
key has become the **index**.

In [3]:
# The intermediate object is lazy - nothing is computed yet
gb = df.groupby("city")
print(type(gb).__name__)
print("ngroups:", gb.ngroups)
print("groups :", {k: v.tolist() for k, v in gb.groups.items()})

DataFrameGroupBy
ngroups: 3
groups : {'Delhi': [2, 4], 'Mumbai': [1, 5], 'Pune': [0, 3]}


## 1.1 ⚠️ `size()` and `count()` are different

In [15]:
d = df.copy()
print(d)
d.loc[2, "python"] = np.nan          # Amit's mark goes missing

print("size()  - counts ROWS:")
print(d.groupby("city").size())
print()
print("count() - counts NON-MISSING values:")
print(d.groupby("city")["python"].count())
print()
print("If those two disagree, you have missing data.")

    name    city grade  python  stats
0   Ravi    Pune     A      88     71
1   Sara  Mumbai     A      91     84
2   Amit   Delhi     C      45     38
3   Neha    Pune     B      67     73
4  Kiran   Delhi     B      72     69
5  Priya  Mumbai     B      58     62
size()  - counts ROWS:
city
Delhi     2
Mumbai    2
Pune      2
dtype: int64

count() - counts NON-MISSING values:
city
Delhi     1
Mumbai    2
Pune      2
Name: python, dtype: int64

If those two disagree, you have missing data.


In [ ]:
# Two ways to count, sorted differently
print("value_counts() - sorted by COUNT:")
print(df["city"].value_counts())
print()
print("groupby().size() - sorted by KEY:")
print(df.groupby("city").size())

# Same numbers, different order. Day 15's value_counts was a groupby all along.

---
# 2. Aggregating

## 2.1 Several columns, several functions

In [16]:
# Several columns at once
df.groupby("city")[["python", "stats"]].mean().round(2)

,python,stats
city,,
Delhi,58.5,53.5
Mumbai,74.5,73.0
Pune,77.5,72.0


In [17]:
# Several functions on one column
df.groupby("city")["python"].agg(["mean", "max", "count"]).round(2)

,mean,max,count
city,,,
Delhi,58.5,72,2
Mumbai,74.5,91,2
Pune,77.5,88,2


In [18]:
# A different function per column - Day 5's dictionary at work
df.groupby("city").agg({"python": "mean", "stats": "max"}).round(2)

,python,stats
city,,
Delhi,58.5,69
Mumbai,74.5,84
Pune,77.5,73


## 2.2 Named aggregation — the version worth defaulting to

In [19]:
df.groupby("city").agg(
    avg_python = ("python", "mean"),
    best_stats = ("stats",  "max"),
    n          = ("name",   "size"),
).round(2)

,avg_python,best_stats,n
city,,,
Delhi,58.5,69,2
Mumbai,74.5,84,2
Pune,77.5,73,2


Each result gets the name **you** choose, so the output reads like a report instead of
`mean, max, size`. It also avoids the awkward two-level column headings that
`agg(["mean", "max"])` produces across several columns.

Available functions include `mean`, `median`, `sum`, `min`, `max`, `std`, `count`, `size`,
`first`, `last`, `nunique`.

## 2.3 The group key becomes the index

In [21]:
result = df.groupby("city")["python"].mean()
print(result)
print("index:", result.index.tolist())
print("is 'city' a column?", "city" in result.reset_index().columns)

# Two ways to get a flat table instead
print()
print(df.groupby("city", as_index=False)["python"].mean().round(2))
print()
print(df.groupby("city")["python"].mean().reset_index().round(2))

city
Delhi     58.5
Mumbai    74.5
Pune      77.5
Name: python, dtype: float64
index: ['Delhi', 'Mumbai', 'Pune']
is 'city' a column? True

     city  python
0   Delhi    58.5
1  Mumbai    74.5
2    Pune    77.5

     city  python
0   Delhi    58.5
1  Mumbai    74.5
2    Pune    77.5


In [ ]:
# Sorted by KEY, not by value - sort the result yourself
print("as returned (alphabetical):")
print(df.groupby("city")["python"].mean().round(2))
print()
print("sorted by value, biggest first:")
print(df.groupby("city")["python"].mean().sort_values(ascending=False).round(2))

---
# 3. Grouping by two columns

In [22]:
long = df.groupby(["city", "grade"])["python"].mean()
print(long)
print()
print("index levels:", long.index.nlevels)

city    grade
Delhi   B        72.0
        C        45.0
Mumbai  A        91.0
        B        58.0
Pune    A        88.0
        B        67.0
Name: python, dtype: float64

index levels: 2


In [23]:
# unstack moves the LAST index level up into the columns
long.unstack()

# The NaNs are combinations that do not exist - no Delhi student got an A.

grade,A,B,C
city,,,
Delhi,NaN,72.0,45.0
Mumbai,91.0,58.0,NaN
Pune,88.0,67.0,NaN


In [24]:
# pivot_table does the same in one step, the way a spreadsheet user thinks
df.pivot_table(index="city", columns="grade", values="python", aggfunc="mean")

grade,A,B,C
city,,,
Delhi,NaN,72.0,45.0
Mumbai,91.0,58.0,NaN
Pune,88.0,67.0,NaN


In [ ]:
# fill_value replaces the empty combinations
df.pivot_table(index="city", columns="grade", values="python",
               aggfunc="mean", fill_value=0)

               

grade,A,B,C
city,,,
Delhi,0.0,72.0,45.0
Mumbai,91.0,58.0,0.0
Pune,88.0,67.0,0.0


In [26]:
print(df)

    name    city grade  python  stats
0   Ravi    Pune     A      88     71
1   Sara  Mumbai     A      91     84
2   Amit   Delhi     C      45     38
3   Neha    Pune     B      67     73
4  Kiran   Delhi     B      72     69
5  Priya  Mumbai     B      58     62


---
# 4. `transform` — put the group answer beside every row

## 4.1 The problem `agg` cannot solve

*"Which students scored above their **own city's** average?"*

In [ ]:
avg = df.groupby("city")["python"].mean()
print("the group averages have", len(avg), "rows")
print("but df has", len(df), "rows - they will not line up")

# agg SHRINKS the table. That is its job.

In [29]:
df["city_avg"] = df.groupby("city")["python"].transform("mean")
df["city_max"] = df.groupby("city")["stats"].transform("max")
df[["name", "city", "python", "city_avg"]]

# Six rows in, six rows out - the group's answer repeated for each member.
print(df)

    name    city grade  python  stats  city_avg  city_max
0   Ravi    Pune     A      88     71      77.5        73
1   Sara  Mumbai     A      91     84      74.5        84
2   Amit   Delhi     C      45     38      58.5        69
3   Neha    Pune     B      67     73      77.5        73
4  Kiran   Delhi     B      72     69      58.5        69
5  Priya  Mumbai     B      58     62      74.5        84


In [30]:
# And now the question is one line
df[df["python"] > df["city_avg"]]["name"].tolist()

['Ravi', 'Sara', 'Kiran']

> **`agg` answers questions ABOUT groups. `transform` answers questions about rows that need
> to know something about their group.** That is the whole distinction.

## 4.2 ⚠️ The pitfall — rows with a missing group key disappear

In [31]:
d = df.copy()
d.loc[0, "city"] = np.nan            # Ravi's city goes missing

counts = d.groupby("city")["python"].count()
print(counts)
print()
print("grouped total:", counts.sum(), "but the table has", len(d), "rows")
print("Ravi is simply not in any group. No error. No warning.")

city
Delhi     2
Mumbai    2
Pune      1
Name: python, dtype: int64

grouped total: 5 but the table has 6 rows
Ravi is simply not in any group. No error. No warning.


In [32]:
# Defence 1 - make the missing group visible
print(d.groupby("city", dropna=False)["python"].count())

city
Delhi     2
Mumbai    2
Pune      1
NaN       1
Name: python, dtype: int64


In [33]:
# Defence 2 - check the arithmetic
sizes = d.groupby("city", dropna=False).size()
print("accounts for every row:", sizes.sum() == len(d))

# Comparing the grouped total against the row count is the cheapest audit there is.

accounts for every row: True


> ### The third silent-missing-data trap of the week
>
> - **Day 14:** `np.nansum` of an all-missing column returns `0.0`
> - **Day 15:** pandas skips `nan` in a mean without telling you
> - **Today:** a missing group key removes the row from the report entirely
>
> The pattern is always the same — the number looks reasonable and nothing complains.
> **Check your counts.**

---
# 5. Putting it together — a department report

In [ ]:
# Build the input, with one missing department and one missing salary
pd.DataFrame({
    "name":   ["Ravi", "Sara", "Amit", "Neha", "Kiran", "Priya"],
    "dept":   ["Sales", "Sales", "IT", "IT", "Sales", None],
    "salary": [52000, 61000, np.nan, 58000, 49000, 55000],
}).to_csv("staff.csv", index=False)

staff = pd.read_csv("staff.csv")

# ---- 0. audit BEFORE grouping
print("rows:", len(staff))
print("missing dept:", staff["dept"].isna().sum())

In [ ]:
# ---- 1. one summary row per department
report = staff.groupby("dept", dropna=False).agg(
    headcount  = ("name",   "size"),
    paid       = ("salary", "count"),      # non-missing salaries
    avg_salary = ("salary", "mean"),
    top_salary = ("salary", "max"),
).round(0)

# ---- 2. the audit that catches dropped rows
assert report["headcount"].sum() == len(staff)

# ---- 3. sort by what matters, not by name
report = report.sort_values("avg_salary", ascending=False)
print(report)

In [ ]:
# ---- 4. who beats their own department average?
staff["dept_avg"] = staff.groupby("dept")["salary"].transform("mean")
print(staff[staff["salary"] > staff["dept_avg"]]["name"].tolist())

In [ ]:
# Proof that the assert earns its place
bad = staff.groupby("dept").agg(headcount=("name", "size"))   # dropna defaults to True
print("without dropna=False, headcount totals",
      bad["headcount"].sum(), "of", len(staff), "rows")
print("Priya has no department, so she vanished from the report.")

Note `headcount` (6 rows) and `paid` (5 salaries) differ — that gap is Amit's missing salary,
and reporting both is what makes the average honest.

---
# 6. Recap — the twelve things to remember

1. `groupby` = split, apply, combine — you write one line.
2. Hear the word **"per"** in a question and reach for `groupby`.
3. The group key becomes the **index** of the result.
4. Results are sorted by **key**, not by value. Sort them yourself.
5. `size()` counts rows; `count()` counts non-missing values.
6. If those two disagree, you have missing data.
7. `agg(["mean", "max"])` for many functions on one column.
8. `agg({"a": "mean"})` for a different function per column.
9. Named aggregation gives your own column names — prefer it.
10. Two keys give a MultiIndex; `unstack` or `pivot_table` widens it.
11. `agg` **shrinks** the table; `transform` keeps the original shape.
12. ⚠️ Rows with a missing group key **vanish**. Use `dropna=False`.

---

### 📝 Now open **`Day17_Practice_Questions.ipynb`** for the 30-minute practice session.

### Homework
- Rewrite Day 15's loop-over-cities as a single `groupby`.
- Build a report with four named aggregations and an `assert` that the counts add up.
- Use `transform` to find every row above its own group's average.

### Next class — Topic 1.17: Merge, join & concat
Combining two tables on a shared key — and what happens to the rows that do not match.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*